**PARCIAL FINAL: ANALITICA DE DATOS**

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.electiva.datos_csv;

In [0]:
%sql

CREATE OR REPLACE TABLE workspace.electiva.customers
USING DELTA
AS SELECT * FROM read_files('/Volumes/workspace/electiva/datos_csv/customers.csv', format => 'csv', header => true,
  inferSchema => true);

CREATE OR REPLACE TABLE workspace.electiva.employee
USING DELTA
AS SELECT * FROM read_files('/Volumes/workspace/electiva/datos_csv/employees.csv',format => 'csv',header => true,
  inferSchema => true);

CREATE OR REPLACE TABLE workspace.electiva.orders
USING DELTA
AS SELECT * FROM read_files('/Volumes/workspace/electiva/datos_csv/orders.csv',format => 'csv',header => true,
  inferSchema => true);

CREATE OR REPLACE TABLE workspace.electiva.product_categories
USING DELTA
AS SELECT * FROM read_files('/Volumes/workspace/electiva/datos_csv/productcategories.csv',format => 'csv',header => true, inferSchema => true);

CREATE OR REPLACE TABLE workspace.electiva.product_sub_categories
USING DELTA
AS SELECT * FROM read_files('/Volumes/workspace/electiva/datos_csv/productsubcategories.csv',format => 'csv',header => true, inferSchema => true);

CREATE OR REPLACE TABLE workspace.electiva.products
USING DELTA
AS SELECT * FROM read_files('/Volumes/workspace/electiva/datos_csv/products.csv',format => 'csv',header => true,
  inferSchema => true);

CREATE OR REPLACE TABLE workspace.electiva.vendor
USING DELTA
AS SELECT * FROM read_files('/Volumes/workspace/electiva/datos_csv/vendors.csv',format => 'csv',header => true,
  inferSchema => true);
  
CREATE OR REPLACE TABLE workspace.electiva.vendor_product
USING DELTA
AS SELECT * FROM read_files('/Volumes/workspace/electiva/datos_csv/vendorproduct.csv',format => 'csv',header => true,
  inferSchema => true);

In [0]:
%sql
SELECT table_name
FROM workspace.information_schema.tables
WHERE table_schema = 'electiva'
AND table_name IN (
    'customers',
    'employee',
    'orders',
    'product_categories',
    'product_sub_categories',
    'products',
    'vendor',
    'vendor_product'
)
ORDER BY table_name;

**Punto #1: Construcción del nombre de usuario**

In [0]:

%sql
SELECT CustomerID AS customerid, CONCAT(FirstName, ' ', LastName) AS fullname, CONCAT(CAST(CustomerID AS STRING),'-',
        UPPER(SUBSTRING(FirstName, 1, 2)), '-', LOWER(SUBSTRING(LastName, 1, 2)), '-AW') AS user_name
FROM workspace.electiva.customers;

_En esta etapa se llevó a cabo la asignación de un nombre de usuario individual para cada uno de los clientes registrados en la base de datos. Para su elaboración, se estableció una estructura específica que combina el identificador correspondiente a cada cliente con las dos letras iniciales de su nombre, escritas en mayúscula, y las dos primeras letras de su apellido, expresadas en minúscula, incorporando al final el sufijo AW. Mediante esta metodología, fue posible organizar la información personal bajo un formato uniforme y previamente definido, permitiendo generar identificadores diferenciados que contribuyen a mejorar los procesos de reconocimiento, búsqueda y administración de los registros dentro del sistema._

**Punto #2: Crear la tabla page_user**

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.electiva.page_user (customerid INT,user_name STRING)
USING DELTA;

INSERT INTO workspace.electiva.page_user
SELECT CustomerID AS customerid, CONCAT(CAST(CustomerID AS STRING), '-', UPPER(SUBSTRING(FirstName, 1, 2)), '-',
LOWER(SUBSTRING(LastName, 1, 2)), '-AW') AS user_name
FROM workspace.electiva.customers;

In [0]:
%sql
SELECT * FROM workspace.electiva.page_user;

_Posteriormente, se procedió con la creación de la tabla denominada page_user, cuya finalidad principal fue centralizar y conservar de manera organizada el código de identificación de cada cliente junto con el nombre de usuario asignado previamente. Una vez definida su estructura, se realizó el proceso de incorporación de la información, obteniendo un total de 635 registros almacenados satisfactoriamente. La implementación de esta tabla independiente permitió consolidar los datos de identificación en un espacio específico de la base de datos, optimizando su disponibilidad y facilitando su recuperación cuando fuera necesario. Asimismo, esta organización contribuye a simplificar las consultas posteriores y a establecer asociaciones entre los registros de los clientes y sus respectivas órdenes de compra._

**Punto #3: Descuentos por categoría – agosto de 2011**

In [0]:
%sql
CREATE OR REPLACE VIEW workspace.electiva.orders_august_2011 AS

SELECT
    p.ProductName AS productname,
    p.ModelName AS modelname,
    pu.user_name,
    o.SalesOrderID AS salesorderid,
    pc.CategoryName AS categoryname,
    o.OrderDate AS orderdate,
    o.LineTotal AS linetotal,

    CASE
        WHEN pc.CategoryName = 'Bikes' THEN 0.10
        WHEN pc.CategoryName = 'Clothing' THEN 0.15
        WHEN pc.CategoryName = 'Components' THEN 0.08
        WHEN pc.CategoryName = 'Accessories' THEN 0.20
        ELSE 0
    END AS `descuento_mitad_año`,

    o.LineTotal - (
        o.LineTotal *
        CASE
            WHEN pc.CategoryName = 'Bikes' THEN 0.10
            WHEN pc.CategoryName = 'Clothing' THEN 0.15
            WHEN pc.CategoryName = 'Components' THEN 0.08
            WHEN pc.CategoryName = 'Accessories' THEN 0.20
            ELSE 0
        END
    ) AS total_descuentos_aplicados

FROM workspace.electiva.orders o

INNER JOIN workspace.electiva.products p
    ON o.ProductID = p.ProductID

INNER JOIN workspace.electiva.product_sub_categories psc
    ON p.SubCategoryID = psc.SubCategoryID

INNER JOIN workspace.electiva.product_categories pc
    ON psc.CategoryID = pc.CategoryID

INNER JOIN workspace.electiva.page_user pu
    ON o.CustomerID = pu.customerid

WHERE YEAR(o.OrderDate) = 2011
  AND MONTH(o.OrderDate) = 8;

In [0]:
%sql
SELECT *
FROM workspace.electiva.orders_august_2011
LIMIT 100;

_Se llevó a cabo la elaboración de una vista en la base de datos con el propósito de consolidar las transacciones comerciales efectuadas durante el mes de agosto de 2011. Para ello, se integraron diferentes fuentes de información, estableciendo relaciones entre las órdenes de compra, los datos de los clientes, los productos adquiridos y las categorías a las que estos pertenecen. Adicionalmente, se incorporaron los porcentajes de descuento definidos para cada clasificación de productos, aplicándolos sobre los valores correspondientes con el fin de determinar el importe definitivo de cada compra una vez efectuadas las respectivas deducciones._
_La información obtenida permitió examinar el comportamiento de los valores de compra en función de las distintas categorías de productos, proporcionando una perspectiva más amplia sobre el efecto que pueden tener las promociones comerciales en las transacciones realizadas por los clientes. De esta manera, la vista constituye una herramienta de apoyo para evaluar estrategias de descuentos estacionales y analizar su posible impacto en las ventas. Como resultado de la consulta, se obtuvieron 1.092 registros que cumplen con el criterio temporal establecido para el periodo de análisis._

**Punto #4: Clasificación de clientes**

In [0]:
%sql
CREATE OR REPLACE MATERIALIZED VIEW workspace.electiva.customer_class AS

WITH ordenes_cliente AS (
    SELECT
        CustomerID,
        COUNT(DISTINCT SalesOrderID) AS amount_of_orders
    FROM workspace.electiva.orders
    WHERE YEAR(OrderDate) = 2011
    GROUP BY CustomerID
)

SELECT
    oc.amount_of_orders,
    pu.user_name,
    CONCAT(c.FirstName, ' ', c.LastName) AS customerfullname,

    CASE
        WHEN oc.amount_of_orders = 3 THEN 'Oro'
        WHEN oc.amount_of_orders = 2 THEN 'Plata'
        WHEN oc.amount_of_orders < 2 THEN 'Ocasional'
    END AS user_level,

    c.CustomerID AS customerid

FROM ordenes_cliente oc

INNER JOIN workspace.electiva.customers c
    ON oc.CustomerID = c.CustomerID

INNER JOIN workspace.electiva.page_user pu
    ON c.CustomerID = pu.customerid;

In [0]:
%sql
SELECT *
FROM workspace.electiva.customer_class
LIMIT 100;

_Se implementó una vista materializada bajo el nombre de customer_class, diseñada para determinar la cantidad de órdenes de compra únicas que registró cada cliente a lo largo del año 2011. Con base en los resultados obtenidos mediante este procedimiento, se definió un sistema de segmentación compuesto por tres grupos, teniendo en cuenta la frecuencia de compra de cada usuario. En primer lugar, se asignó la categoría Oro a aquellos clientes que efectuaron tres órdenes durante el periodo evaluado; la clasificación Plata se destinó a quienes registraron dos órdenes, mientras que aquellos con una frecuencia inferior a dos compras fueron ubicados en el segmento Ocasional._

_La aplicación de este mecanismo de segmentación permite reconocer las diferencias existentes en los hábitos de consumo y en el nivel de participación comercial de los clientes. Asimismo, proporciona una base de información relevante para que la empresa pueda estructurar acciones de mercadeo más específicas, desarrollar programas de incentivos y establecer mecanismos de fidelización que contribuyan a fortalecer la relación con sus consumidores y favorecer el crecimiento de las ventas._

_El desarrollo de los cuatro ejercicios permitió comprender y aplicar la utilidad del lenguaje SQL en el entorno Databricks como una herramienta fundamental para la gestión y el análisis de información empresarial. Mediante la creación de tablas, la elaboración de consultas y la implementación de vistas convencionales y materializadas, fue posible organizar, relacionar y procesar diferentes conjuntos de datos, obteniendo información relevante sobre los clientes, sus hábitos de compra y el comportamiento de las ventas._

_Las actividades desarrolladas también permitieron reconocer que la administración de bases de datos va más allá del almacenamiento de información, puesto que constituye un recurso estratégico para interpretar el comportamiento del mercado, identificar patrones de consumo y generar conocimiento de valor para una organización. Así, la aplicación de herramientas como SQL y Databricks fortalece los procesos de análisis, facilita la toma de decisiones fundamentadas y proporciona elementos para diseñar estrategias comerciales que contribuyan al crecimiento y la competitividad empresarial._